In [85]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from langdetect import detect
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
from scikeras.wrappers import KerasClassifier
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from lime.lime_text import LimeTextExplainer
#from imblearn.over_sampling import SMOTE




[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [56]:
# load the dataset
df = pd.read_csv('judge-1377884607_tweet_product_company.csv', encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [57]:
# check info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [58]:
# check for duplicates
df.duplicated().sum()

22

In [59]:
# check for missing values
df.isna().sum()

tweet_text                                               1
emotion_in_tweet_is_directed_at                       5802
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [60]:
# drop duplicates
df.drop_duplicates()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion
...,...,...,...
9088,Ipad everywhere. #SXSW {link},iPad,Positive emotion
9089,"Wave, buzz... RT @mention We interrupt your re...",NaN,No emotion toward brand or product
9090,"Google's Zeiger, a physician never reported po...",NaN,No emotion toward brand or product
9091,Some Verizon iPhone customers complained their...,NaN,No emotion toward brand or product


In [61]:
# drop the emotion_in_tweet_is_directed_at as it contains much missing values
df.drop(columns='emotion_in_tweet_is_directed_at', inplace=True)

In [62]:
# rename columns
df = df.rename(columns={'tweet_text':'text','is_there_an_emotion_directed_at_a_brand_or_product': 'sentiment'})

In [63]:
# create another dataframe caaled df_sentiment
df_sentiment = df.copy()
df_sentiment

,text,sentiment
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion
...,...,...
9088,Ipad everywhere. #SXSW {link},Positive emotion
9089,"Wave, buzz... RT @mention We interrupt your re...",No emotion toward brand or product
9090,"Google's Zeiger, a physician never reported po...",No emotion toward brand or product
9091,Some Verizon iPhone customers complained their...,No emotion toward brand or product


In [64]:
# check the value counts of sentiment
df_sentiment['sentiment'].value_counts()

sentiment
No emotion toward brand or product    5389
Positive emotion                      2978
Negative emotion                       570
I can't tell                           156
Name: count, dtype: int64

In [73]:
# remove whitespaces and convert the text column to string
df_sentiment['text'] = df_sentiment['text'].fillna('').astype(str)

# do feature engineering to find out the number of words, characters and sentences
df_sentiment['words'] = df_sentiment['text'].apply(len)
df_sentiment['chars'] = df_sentiment['text'].apply(lambda text: len(nltk.word_tokenize(text)))
df_sentiment['sent'] = df_sentiment['text'].apply(lambda text: len(nltk.sent_tokenize(text)))

df_sentiment

,text,sentiment,words,chars,sent
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion,127,32,5
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion,139,29,3
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion,79,20,2
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion,82,21,2
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion,131,29,1
...,...,...,...,...,...
9088,Ipad everywhere. #SXSW {link},Positive emotion,29,8,2
9089,"Wave, buzz... RT @mention We interrupt your re...",No emotion toward brand or product,125,26,1
9090,"Google's Zeiger, a physician never reported po...",No emotion toward brand or product,145,32,3
9091,Some Verizon iPhone customers complained their...,No emotion toward brand or product,140,26,2


In [74]:
# check for descriptive statistics
df_sentiment.describe()

,words,chars,sent
count,9093.000000,9093.000000,9093.000000
mean,104.950731,24.412295,1.880018
std,27.208419,6.505483,0.943527
min,0.000000,0.000000,0.000000
25%,86.000000,20.000000,1.000000
50%,109.000000,25.000000,2.000000
75%,126.000000,29.000000,2.000000
max,178.000000,49.000000,7.000000


In [77]:
# label encode the target
le = LabelEncoder()
df_sentiment['sentiment'] = le.fit_transform(df_sentiment['sentiment'])
df_sentiment['sentiment'] 


0       1
1       3
2       3
3       1
4       3
       ..
9088    3
9089    2
9090    2
9091    2
9092    2
Name: sentiment, Length: 9093, dtype: int32

In [ ]:

class SentimentPreprocessor:
    def __init__(self, text_column, target_column):
        self.text_column = text_column
        self.target_column = target_column
        self.vectorizer = TfidfVectorizer()
        self.stop_words = set(stopwords.words('english'))

    def _is_english(self, text):
        try:
            return detect(text) == 'en'
        except:
            return False

    def _remove_emojis(self, text):
        emoji_pattern = re.compile("[" 
                                   u"\U0001F600-\U0001F64F"  # emoticons
                                   u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                                   u"\U0001F680-\U0001F6FF"  # transport & map symbols
                                   u"\U0001F1E0-\U0001F1FF"  # flags
                                   "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', text)

    def _clean_text(self, text):
        text = self._remove_emojis(text)
        text = re.sub(r'\d+', '', text) 
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        tokens = word_tokenize(text)
        tokens = [word for word in tokens if word not in self.stop_words and word.isalpha()]
        return " ".join(tokens)

    def preprocess(self, df_sentiment):
       print("Removing non-English rows")
       df_sentiment = df_sentiment[df_sentiment[self.text_column].apply(self._is_english)]

       print("Cleaning and tokenizing text")
       df_sentiment.loc[:, self.text_column] = df_sentiment[self.text_column].astype(str).apply(self._clean_text)

       print("Vectorizing with TF-IDF")
       X_tfidf = self.vectorizer.fit_transform(df_sentiment[self.text_column])
       y = df_sentiment[self.target_column].values

       return X_tfidf, y


    def split_and_process(self, df_sentiment, test_size=0.2):
        X, y = self.preprocess(df_sentiment)
        return train_test_split(X, y, test_size=test_size, random_state=42)


# Example usage
processor = SentimentPreprocessor(text_column='text', target_column='sentiment')
X_train, X_test, y_train, y_test = processor.split_and_process(df_sentiment)


Removing non-English rows
Cleaning and tokenizing text
Vectorizing with TF-IDF


In [89]:
# create an oop function to do data preprocessing

class SentimentPreprocessor:
    def __init__(self, text_column, target_column):
        self.text_column = text_column
        self.target_column = target_column
        self.vectorizer = TfidfVectorizer()
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def _is_english(self, text):
        try:
            return detect(text) == 'en'
        except:
            return False

    def _remove_emojis(self, text):
        emoji_pattern = re.compile("[" 
                                   u"\U0001F600-\U0001F64F"
                                   u"\U0001F300-\U0001F5FF"
                                   u"\U0001F680-\U0001F6FF"
                                   u"\U0001F1E0-\U0001F1FF"
                                   "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', text)

    def _clean_text(self, text):
        text = self._remove_emojis(text)
        text = re.sub(r'\d+', '', text)
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        tokens = word_tokenize(text)
        tokens = [
            self.lemmatizer.lemmatize(word)
            for word in tokens
            if word.isalpha() and word not in self.stop_words
        ]
        return tokens

    def preprocess(self, df_sentiment):
        print("Removing non-English rows")
        df_sentiment = df_sentiment[df_sentiment[self.text_column].astype(str).apply(self._is_english)].copy()

        print("Cleaning text, lemmatizing, and tokenizing")
        df_sentiment['tokens'] = df_sentiment[self.text_column].astype(str).apply(self._clean_text)
        df_sentiment['clean_text'] = df_sentiment['tokens'].apply(lambda x: ' '.join(x))

        print("Vectorizing 'clean_text' with TF-IDF")
        X_tfidf = self.vectorizer.fit_transform(df_sentiment['clean_text'])
        y = df_sentiment[self.target_column].values

        print("Preview of cleaned data:")
        print(df_sentiment[['text', 'clean_text', 'tokens']].head())

        return df_sentiment, X_tfidf, y

    def split_and_process(self, df_sentiment, test_size=0.2):
        df_cleaned, X, y = self.preprocess(df_sentiment)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
        return df_cleaned, X_train, X_test, y_train, y_test



In [90]:
# apply the class on the data

processor = SentimentPreprocessor(text_column='text', target_column='sentiment')
df_cleaned, X_train, X_test, y_train, y_test = processor.split_and_process(df_sentiment)

print(df_cleaned[['text', 'clean_text', 'tokens']].head())


Removing non-English rows
Cleaning text, lemmatizing, and tokenizing
Vectorizing 'clean_text' with TF-IDF
Preview of cleaned data:
                                                text  \
0  .@wesley83 I have a 3G iPhone. After 3 hrs twe...   
1  @jessedee Know about @fludapp ? Awesome iPad/i...   
2  @swonderlin Can not wait for #iPad 2 also. The...   
3  @sxsw I hope this year's festival isn't as cra...   
4  @sxtxstate great stuff on Fri #SXSW: Marissa M...   

                                          clean_text  \
0  wesley g iphone hr tweeting riseaustin dead ne...   
1  jessedee know fludapp awesome ipadiphone app y...   
2                swonderlin wait ipad also sale sxsw   
3  sxsw hope year festival isnt crashy year iphon...   
4  sxtxstate great stuff fri sxsw marissa mayer g...   

                                              tokens  
0  [wesley, g, iphone, hr, tweeting, riseaustin, ...  
1  [jessedee, know, fludapp, awesome, ipadiphone,...  
2         [swonderlin, wait, i

In [66]:

# cross validate different models

# Decision trees
dt_pipeline = Pipeline([
    ('clf', DecisionTreeClassifier(random_state=42))
])

dt_param_grid = {
    'clf__max_depth': [10, 20, 30, None],
    'clf__min_samples_split': [2, 5, 10],
    'clf__criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(dt_pipeline, dt_param_grid, cv=5, scoring='accuracy', n_jobs=1)
dt_grid.fit(X_train, y_train)





GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('clf',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=1,
             param_grid={'clf__criterion': ['gini', 'entropy'],
                         'clf__max_depth': [10, 20, 30, None],
                         'clf__min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [ ]:
# Random Forest 
rf_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

rf_param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
    'clf__criterion': ['gini', 'entropy']
}

rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=5, scoring='accuracy', n_jobs=1)
rf_grid.fit(X_train, y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('clf',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=1,
             param_grid={'clf__criterion': ['gini', 'entropy'],
                         'clf__max_depth': [None, 10, 20],
                         'clf__min_samples_split': [2, 5],
                         'clf__n_estimators': [100, 200]},
             scoring='accuracy')

In [ ]:
# Keras classifier

# Define simple model builder function
def build_model(hidden_units=100, activation='relu', alpha=0.0001):
    model = Sequential([
        Dense(hidden_units, input_shape=(X_train.shape[1],), activation=activation),
        Dense(len(np.unique(y_train)), activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Wrap with KerasClassifier
kc = KerasClassifier(model=build_model, verbose=0, epochs=10)

# Put in pipeline
pipeline = Pipeline([
    ('clf', kc)
])

# Parameter grid
param_grid = {
    'clf__model__hidden_units': [50, 100],
    'clf__model__activation': ['relu', 'tanh'],
    'clf__model__alpha': [0.0001, 0.001]
}

# Run Grid Search
kc_grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
kc_grid.fit(X_train, y_train)

c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regulariz

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('clf',
                                        KerasClassifier(epochs=10, model=<function build_model at 0x000002685AE3FD90>, verbose=0))]),
             param_grid={'clf__model__activation': ['relu', 'tanh'],
                         'clf__model__alpha': [0.0001, 0.001],
                         'clf__model__hidden_units': [50, 100]},
             scoring='accuracy')

In [ ]:
# evaluate the best perfoming model

models = {
    "Decision Tree": dt_grid,
    "Random Forest": rf_grid,
    "Neural Network": kc_grid
}

for name, model in models.items():
    print(f"{name} Results")
    print("Best Params:", model.best_params_)
    print("Best Cross-Validated Accuracy:", model.best_score_)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    print("Classification Report:", classification_report(y_test, y_pred_test))


Decision Tree Results
Best Params: {'clf__criterion': 'gini', 'clf__max_depth': 10, 'clf__min_samples_split': 2}
Best Cross-Validated Accuracy: 0.6192387543252595
Classification Report:                                     precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        28
                  Negative emotion       0.86      0.05      0.10       115
No emotion toward brand or product       0.65      0.83      0.73      1056
                  Positive emotion       0.54      0.41      0.47       608

                          accuracy                           0.63      1807
                         macro avg       0.51      0.32      0.32      1807
                      weighted avg       0.62      0.63      0.59      1807

Random Forest Results
Best Params: {'clf__criterion': 'gini', 'clf__max_depth': None, 'clf__min_samples_split': 2, 'clf__n_estimators': 100}
Best Cross-Validated Accuracy: 0.6635294117647058


c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  sample_weight=sample_weight,
c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  sample_weight=sample_weight,
c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  sample_weight=sample_weight,


Classification Report:                                     precision    recall  f1-score   support

                      I can't tell       0.17      0.04      0.06        28
                  Negative emotion       0.66      0.17      0.26       115
No emotion toward brand or product       0.68      0.90      0.78      1056
                  Positive emotion       0.71      0.43      0.53       608

                          accuracy                           0.68      1807
                         macro avg       0.55      0.38      0.41      1807
                      weighted avg       0.68      0.68      0.65      1807

Neural Network Results
Best Params: {'clf__model__activation': 'relu', 'clf__model__alpha': 0.001, 'clf__model__hidden_units': 50}
Best Cross-Validated Accuracy: 0.6502422145328719
Classification Report:                                     precision    recall  f1-score   support

                      I can't tell       0.17      0.07      0.10        28
         